In [6]:
"""
Logistics Data Cleaning and Preprocessing Pipeline
Task: Week 2 - Data Collection, Cleaning, and Preprocessing
"""

import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Simulate Raw Logistics Dataset
np.random.seed(42)
n_records = 150

raw_data = {
    "Shipment_ID": [f"SHP_{1000+i}" for i in range(n_records)],
    "Origin": np.random.choice(
        ["Delhi", "Mumbai", "Bangalore", "Kolkata"], n_records
    ),
    "Destination": np.random.choice(
        ["Chennai", "Hyderabad", "Pune", "Jaipur"], n_records
    ),
    "Vehicle_Type": np.random.choice(
        ["Truck", "Van", "Air", None], n_records, p=[0.5, 0.3, 0.15, 0.05]
    ),
    "Distance_KM": np.random.normal(loc=850, scale=300, size=n_records).round(
        1
    ),
    "Shipping_Cost": np.random.normal(
        loc=4200, scale=1200, size=n_records
    ).round(2),
    "Transit_Days": np.random.choice(
        [1, 2, 3, 4, 5, -2, 45, np.nan],
        n_records,
        p=[0.3, 0.3, 0.2, 0.1, 0.04, 0.02, 0.02, 0.02],
    ),
    "Delivery_Status": np.random.choice(
        ["Delivered", "In Transit", "Delayed", "delivered "], n_records
    ),
}

df = pd.DataFrame(raw_data)

# Inject deliberate operational data anomalies
df.loc[10, "Distance_KM"] = -50.0
df.loc[25, "Shipping_Cost"] = np.nan

print("--- Initial Data Summary ---")
print(df.info())

# Save raw dataset to disk
os.makedirs("data", exist_ok=True)
df.to_csv("data/raw_logistics_data.csv", index=False)

# 2. String Cleaning & Text Standardization
df["Delivery_Status"] = (
    df["Delivery_Status"].astype(str).str.strip().str.title()
)
df["Vehicle_Type"] = (
    df["Vehicle_Type"].fillna("Unknown").str.strip().str.title()
)

# 3. Filtering Illogical Values (Domain Sanity Checks)
df = df[df["Distance_KM"] > 0]
df = df[(df["Transit_Days"].isna()) | (df["Transit_Days"] > 0)]

# 4. Imputing Missing Values
# Impute Vehicle_Type using mode
mode_vehicle = df[df["Vehicle_Type"] != "Unknown"]["Vehicle_Type"].mode()[0]
df["Vehicle_Type"] = df["Vehicle_Type"].replace("Unknown", mode_vehicle)

# Impute Shipping_Cost using median
cost_median = df["Shipping_Cost"].median()
df["Shipping_Cost"] = df["Shipping_Cost"].fillna(cost_median)

# Impute missing Transit_Days using median
transit_median = df["Transit_Days"].median()
df["Transit_Days"] = df["Transit_Days"].fillna(transit_median)

# 5. Outlier Detection & Capping (IQR Winsorization)
Q1 = df["Transit_Days"].quantile(0.25)
Q3 = df["Transit_Days"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + (1.5 * IQR)

df["Transit_Days_Capped"] = np.where(
    df["Transit_Days"] > upper_bound, upper_bound, df["Transit_Days"]
)

# 6. Feature Normalization (Min-Max Scaling)
scaler = MinMaxScaler()
df[["Distance_Scaled", "Cost_Scaled"]] = scaler.fit_transform(
    df[["Distance_KM", "Shipping_Cost"]]
)

# Save cleaned output
df.to_csv("data/cleaned_logistics_data.csv", index=False)

print("\n--- Cleaned & Processed Data (First 5 Rows) ---")
print(
    df[
        [
            "Shipment_ID",
            "Vehicle_Type",
            "Transit_Days_Capped",
            "Distance_Scaled",
            "Cost_Scaled",
            "Delivery_Status",
        ]
    ].head()
)
print("\nCleaned records successfully saved to data/cleaned_logistics_data.csv")

--- Initial Data Summary ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Shipment_ID      150 non-null    object 
 1   Origin           150 non-null    object 
 2   Destination      150 non-null    object 
 3   Vehicle_Type     144 non-null    object 
 4   Distance_KM      150 non-null    float64
 5   Shipping_Cost    149 non-null    float64
 6   Transit_Days     147 non-null    float64
 7   Delivery_Status  150 non-null    object 
dtypes: float64(3), object(5)
memory usage: 9.5+ KB
None

--- Cleaned & Processed Data (First 5 Rows) ---
  Shipment_ID Vehicle_Type  Transit_Days_Capped  Distance_Scaled  Cost_Scaled  \
0    SHP_1000          Air                  1.0         0.491300     0.033087   
1    SHP_1001        Truck                  2.0         0.325846     0.330093   
2    SHP_1002        Truck                  2.0         0.